# Module 03: CodeAgent vs ToolCallingAgent

smolagents ships two agent architectures that differ fundamentally in *how* they reason:

| Agent | How it thinks | Execution |
|---|---|---|
| `CodeAgent` | **Thinks in Python** | Generates Python code, executes it in a sandbox, observes stdout / return value |
| `ToolCallingAgent` | **Thinks in JSON** | Generates `{"tool_name": ..., "arguments": {...}}`, framework dispatches to the matching tool, observes return value |

**CodeAgent** can perform arbitrary computation between tool calls — loops, math, string ops — because it is writing and running real Python. The trade-off is an arbitrary code execution surface that must be sandboxed in production.

**ToolCallingAgent** can only call tools by name. There is no code execution. The call log is structured and auditable, making it well-suited for production systems where predictability and security matter.

> **Key question: Which should I use?** — that is exactly what this module answers.

We will give both agents the **same tool** and the **same task**, then compare their internal reasoning steps side by side.

## Setup

In [ ]:
# Install required packages
# Uncomment the line below if running in Google Colab or a fresh environment
# !uv pip install smolagents python-dotenv duckduckgo-search mlflow
# Or using pip:
# !pip install smolagents python-dotenv duckduckgo-search mlflow

In [ ]:
import os

# ----- HF_TOKEN Setup -----
# Option A: Load from .env file (local development)
# from dotenv import load_dotenv
# load_dotenv()

# Option B: Google Colab Secrets
# Uncomment the lines below when running in Google Colab.
# Go to: Colab → Secrets (🔑 icon) → Add HF_TOKEN
# from google.colab import userdata
# os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# Option C: Set directly (not recommended for shared notebooks)
# os.environ['HF_TOKEN'] = 'hf_your_token_here'

In [ ]:
import os
from dotenv import load_dotenv
from smolagents import CodeAgent, ToolCallingAgent, InferenceClientModel, tool

load_dotenv()

model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct",
    token=os.environ["HF_TOKEN"],
)
print("Ready.")

## A Shared Tool for Fair Comparison

To fairly compare both agent types, we'll give them the same tool and the same task.

In [ ]:
@tool
def word_count(text: str) -> int:
    """
    Counts the number of words in the given text string.

    Args:
        text: The input text to count words in.
    """
    return len(text.split())

# Verify it works standalone
print(word_count("The quick brown fox jumps over the lazy dog"))

## CodeAgent: Thinks in Python

Watch the output carefully — you'll see the agent *write Python code* to solve the task, then execute it.

In [ ]:
task = (
    "Count the words in this sentence: "
    "'The quick brown fox jumps over the lazy dog'. "
    "Then double that number and tell me the result."
)

code_agent = CodeAgent(tools=[word_count], model=model)
code_result = code_agent.run(task)

print("\n" + "="*50)
print("CodeAgent result:", code_result)
print("Steps taken:", len(code_agent.memory.steps))

## ToolCallingAgent: Thinks in JSON

Same task, same tool, same model. But this agent emits *structured JSON tool calls* instead of Python code.

In [ ]:
tool_agent = ToolCallingAgent(tools=[word_count], model=model)
tool_result = tool_agent.run(task)

print("\n" + "="*50)
print("ToolCallingAgent result:", tool_result)
print("Steps taken:", len(tool_agent.memory.steps))

## Comparing the Steps

Let's look at what each agent actually did internally.

In [ ]:
print("=== CodeAgent Steps ===")
for i, step in enumerate(code_agent.memory.steps):
    print(f"  [{i}] {type(step).__name__}: {str(step)[:200]}")

print("\n=== ToolCallingAgent Steps ===")
for i, step in enumerate(tool_agent.memory.steps):
    print(f"  [{i}] {type(step).__name__}: {str(step)[:200]}")

## When to Use Each

### Use `CodeAgent` when:
- Your task requires computation, loops, or data manipulation between tool calls
- You need the agent to do math, string processing, or list operations on tool results
- You trust the execution environment (internal tools, trusted infra)
- The task is open-ended and the solution path isn't predictable

### Use `ToolCallingAgent` when:
- Tool calls are the primary action — the agent just needs to dispatch to tools
- You need structured, auditable logs of exactly what was called
- Security matters — no code execution surface
- Your LLM supports native function/tool calling (most modern models do)
- You're building production systems where predictability > flexibility

## Model Agnosticism: Swap to OpenAI

> **Optional** — requires `OPENAI_API_KEY` in your `.env` file.

This is one of smolagents' biggest strengths: the *agent code doesn't change when you swap the model*. You change one line.

In [ ]:
# ── OPTIONAL: requires OPENAI_API_KEY in .env ─────────────────────────────
# Uncomment and run this cell only if you have an OpenAI API key.
# Notice: the agent code below is IDENTICAL to the cells above.

# from smolagents import LiteLLMModel
# 
# openai_model = LiteLLMModel(
#     model_id="gpt-4o-mini",
#     api_key=os.environ.get("OPENAI_API_KEY"),
# )
# 
# # EXACT same agent API — only the model changes
# openai_agent = ToolCallingAgent(tools=[word_count], model=openai_model)
# result = openai_agent.run(task)
# print("OpenAI-backed result:", result)
# 
# # You could also swap to Anthropic:
# # LiteLLMModel(model_id="anthropic/claude-3-5-sonnet-latest", api_key=...)
# # Or Ollama (local, free):
# # LiteLLMModel(model_id="ollama_chat/llama3.2", api_base="http://localhost:11434")
# ──────────────────────────────────────────────────────────────────────────
print("Skipping optional OpenAI demo — uncomment above to run with OPENAI_API_KEY")

## Exercises

In [ ]:
# TODO Exercise 1: Vowel counting comparison
# 1. Write a @tool called `count_vowels` that counts vowels in a string
#    (a, e, i, o, u — case insensitive)
# 2. Run the task "Count the vowels in: 'HuggingFace builds amazing open source tools'"
#    with BOTH CodeAgent and ToolCallingAgent
# 3. Compare the intermediate steps:
#    - Which agent wrote code? Which emitted JSON?
#    - Which produced cleaner, more readable steps?
#    - Did they get the same answer?

# Your code here:

In [ ]:
# TODO Exercise 2: Loop task comparison
# Design a task that requires a loop: e.g., 
# "For each word in ['python', 'data', 'engineer', 'agent'], count its letters
#  and return the word with the most letters."
#
# 1. Try ToolCallingAgent first. Does it handle the iteration well?
# 2. Then try CodeAgent. What's different?
# 3. Write 2–3 sentences explaining which agent type is better suited for this
#    task and why.

# Your code here:

## What You Built

✅ **You now know how to:**
- Understand how `CodeAgent` executes Python to reason between tool calls
- Understand how `ToolCallingAgent` emits JSON dispatch calls
- Compare both agent types on the same task and read the steps
- Apply the decision framework: compute-heavy → CodeAgent; dispatch-heavy → ToolCallingAgent
- Swap any LLM backend with 1–2 lines using `LiteLLMModel`

**Key insight:** smolagents is model-agnostic by design. The same agent code works with HuggingFace, OpenAI, Anthropic, Ollama, and 100+ other providers via LiteLLM.

## Next Module Preview

➡️ **Module 04: Web Search & Browsing**

Your agents have been working with static tools. In Module 04 you'll connect them to the live internet using `DuckDuckGoSearchTool` and `VisitWebpageTool` — giving your agent access to real-time information.